# Hull–White Callable Bond Valuation

This notebook implements a fixed-income pricing and calibration workflow using the **one-factor Hull–White short-rate model**.

### Workflow

1. Load market data from `CallableBond.xlsx`
2. Build the discount/yield term structure
3. Process and interpolate the normal-volatility surface
4. Construct swaption calibration instruments
5. Calibrate Hull–White mean reversion and volatility
6. Run a fixed-mean-reversion calibration as a diagnostic
7. Price a callable fixed-rate bond using a Hull–White tree
8. Price the corresponding non-callable bond
9. Perform an interest-rate sensitivity analysis

> **Data requirement:** Place `CallableBond.xlsx` in the same directory as this notebook (or inside a `data/` subdirectory). No absolute/local machine paths are used.

> **Attribution:** This notebook is a cleaned, refactored, and modernized adaptation of the publicly available `rstreppa/valuation-callables-HullWhite` notebook. The original implementation is acknowledged here so that the repository does not present adapted source code as entirely original work.


In [ ]:
# Core dependencies
from pathlib import Path
from collections import namedtuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from openpyxl import load_workbook
from scipy.interpolate import RegularGridInterpolator
import QuantLib as ql

%matplotlib inline

plt.rcParams["figure.figsize"] = (10, 6)


## 0. Configuration and data loading

The original notebook referenced a machine-specific Windows path. The project version below uses a relative path so that the notebook can be executed after cloning the repository.


In [ ]:
# Locate the workbook without relying on a machine-specific absolute path.
DATA_CANDIDATES = [
    Path("CallableBond.xlsx"),
    Path("data") / "CallableBond.xlsx",
]

data_file = next((path for path in DATA_CANDIDATES if path.is_file()), None)

if data_file is None:
    expected = "\n".join(f"  - {path}" for path in DATA_CANDIDATES)
    raise FileNotFoundError(
        "CallableBond.xlsx was not found. Place the data file in one of these locations:\n"
        f"{expected}"
    )

wb = load_workbook(filename=data_file, read_only=True, data_only=True)

print(f"Loaded market-data workbook: {data_file}")
print(f"Available worksheets: {wb.sheetnames}")


## 1. Utility functions

In [ ]:
def load_workbook_range(range_string, worksheet):
    """Load an Excel range whose first row contains column headers."""
    rows = list(worksheet[range_string])
    if not rows:
        raise ValueError(f"Excel range '{range_string}' is empty.")

    values = [[cell.value for cell in row] for row in rows]
    columns = values[0]

    if any(column is None for column in columns):
        raise ValueError(f"Excel range '{range_string}' contains missing column headers.")

    return pd.DataFrame(values[1:], columns=columns)


def convert_date(date_value):
    """Convert a pandas/Python date-like value to a QuantLib Date."""
    return ql.Date(date_value.day, date_value.month, date_value.year)


def plot_curve(curve, title="Yield Curve"):
    """Plot continuous zero rates from a QuantLib term structure."""
    times = np.linspace(0.0, 15.0, 400)
    rates = [
        curve.zeroRate(float(t), ql.Continuous).rate()
        for t in times
    ]

    fig, ax = plt.subplots()
    ax.plot(times, rates)
    ax.set_xlabel("Tenor (years)")
    ax.set_ylabel("Zero rate")
    ax.set_title(title)
    ax.grid(alpha=0.25)
    plt.show()


def plot_surface(surface_df, title="Volatility Surface"):
    """Plot a rectangular volatility surface using Matplotlib's 3D axes."""
    x = np.asarray(surface_df.index, dtype=float)
    y = np.asarray(surface_df.columns, dtype=float)
    X, Y = np.meshgrid(y, x)
    Z = surface_df.to_numpy(dtype=float)

    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(X, Y, Z, linewidth=0, antialiased=True)

    ax.set_xlabel("Swap tenor (years)")
    ax.set_ylabel("Option tenor (years)")
    ax.set_zlabel("Volatility")
    ax.set_title(title)
    plt.show()


In [ ]:
def make_surface_interpolator(surface_df):
    """Create a linear interpolator for a rectangular volatility surface."""
    option_tenors = np.asarray(surface_df.index, dtype=float)
    swap_tenors = np.asarray(surface_df.columns, dtype=float)
    values = surface_df.to_numpy(dtype=float)

    return RegularGridInterpolator(
        (option_tenors, swap_tenors),
        values,
        method="linear",
        bounds_error=True,
    )


def get_surface_value(interpolator, option_tenor, swap_tenor):
    """Evaluate the volatility interpolator at one option/swap tenor pair."""
    point = np.array([[float(option_tenor), float(swap_tenor)]])
    return float(interpolator(point)[0])


In [ ]:
def create_swaption_helpers(calibration_data, index, term_structure, engine):
    """Create QuantLib swaption calibration helpers from calibration inputs."""
    fixed_leg_tenor = ql.Period(6, ql.Months)
    fixed_leg_day_counter = ql.Thirty360(ql.Thirty360.BondBasis)
    floating_leg_day_counter = ql.Actual360()

    swaptions = [
        ql.SwaptionHelper(
            ql.Period(swap.start, ql.Years),
            ql.Period(swap.length, ql.Years),
            ql.QuoteHandle(ql.SimpleQuote(swap.volatility)),
            index,
            fixed_leg_tenor,
            fixed_leg_day_counter,
            floating_leg_day_counter,
            term_structure,
            ql.CalibrationHelper.RelativePriceError,
        )
        for swap in calibration_data
    ]

    for swaption in swaptions:
        swaption.setPricingEngine(engine)

    return swaptions


def calibration_report(swaptions, calibration_data):
    """Return model-vs-market calibration diagnostics."""
    report = []

    for swaption, market in zip(swaptions, calibration_data):
        model_price = swaption.modelValue()
        market_price = swaption.marketValue()

        implied_vol = swaption.impliedVolatility(
            model_price,
            0.01,     # accuracy
            1000,     # max evaluations
            0.001,    # min vol
            0.9,      # max vol
        )

        report.append(
            {
                "Option Tenor": market.start,
                "Swap Tenor": market.length,
                "Market Vol": market.volatility,
                "Model Price": model_price,
                "Market Price": market_price,
                "Abs Error Price": market_price - model_price,
                "Rel Error Price": (model_price - market_price) / market_price,
                "Implied Vol": implied_vol,
                "Abs Error Vol": implied_vol - market.volatility,
                "Rel Error Vol": (implied_vol - market.volatility) / market.volatility,
            }
        )

    report_df = pd.DataFrame(report)

    cumulative_error = np.sqrt(
        np.sum(report_df["Rel Error Vol"].to_numpy() ** 2)
    )

    print(f"Cumulative relative-volatility error: {cumulative_error:.6f}")
    return report_df


def price_callable_bond(term_structure_handle, mean_reversion, volatility,
                        grid_points, bond):
    """Price a callable fixed-rate bond using a Hull–White tree."""
    model = ql.HullWhite(
        term_structure_handle,
        float(mean_reversion),
        float(volatility),
    )
    engine = ql.TreeCallableFixedRateBondEngine(model, int(grid_points))
    bond.setPricingEngine(engine)
    return bond.dirtyPrice()


## 2. Dates, conventions, and global model settings

In [ ]:
issue_date = ql.Date(10, ql.September, 2015)
valuation_date = ql.Date(31, ql.March, 2017)
maturity_date = ql.Date(10, ql.September, 2045)
first_call_date = ql.Date(10, ql.September, 2020)

curve_day_counter = ql.Actual360()
bond_day_counter = ql.Thirty360(ql.Thirty360.BondBasis)
business_day_convention = ql.ModifiedFollowing
calendar = ql.UnitedStates(ql.UnitedStates.GovernmentBond)

ql.Settings.instance().evaluationDate = valuation_date


## 3. Yield curve and volatility surface

In [ ]:
ws = wb["Market Data"]

# Discount factors
discount_df = load_workbook_range("B19:C44", ws)
discount_df = discount_df.set_index("DATE").dropna()

yield_curve = ql.YieldTermStructureHandle(
    ql.DiscountCurve(
        [convert_date(date) for date in discount_df.index],
        discount_df["DISCOUNT FACTOR"].to_numpy(dtype=float),
        curve_day_counter,
    )
)

plot_curve(yield_curve, title="Input Discount-Curve Zero Rates")


In [ ]:
# Mapping from quoted labels to year fractions used by the source dataset.
tenor_map = {
    "1Mo": 1 / 12,
    "3Mo": 3 / 12,
    "6Mo": 6 / 12,
    "9Mo": 9 / 12,
    "1Yr": 1,
    "2Yr": 2,
    "3Yr": 3,
    "4Yr": 4,
    "5Yr": 5,
    "6Yr": 6,
    "7Yr": 7,
    "8Yr": 8,
    "9Yr": 9,
    "10Yr": 10,
    "12Yr": 12,
    "15Yr": 15,
    "20Yr": 20,
    "25Yr": 25,
    "30Yr": 30,
}

vol_surface = load_workbook_range("I2:U21", ws)
vol_surface = vol_surface.set_index("VOL.MATRIX").dropna()

# Source volatility is quoted in basis points.
vol_surface = vol_surface / 10000.0
vol_surface.columns = [int(str(column)[:-2]) for column in vol_surface.columns]
vol_surface.index = [tenor_map[label] for label in vol_surface.index]

# The source project approximates the lognormal-equivalent volatility by
# normal volatility divided by the corresponding forward rate.
# This approximation is retained from the original methodology.
for option_tenor in vol_surface.index:
    for swap_tenor in vol_surface.columns:
        forward_rate = yield_curve.forwardRate(
            float(option_tenor),
            float(option_tenor + swap_tenor),
            ql.Continuous,
            ql.Annual,
            True,
        ).rate()

        if np.isclose(forward_rate, 0.0):
            raise ZeroDivisionError(
                f"Forward rate is approximately zero at "
                f"({option_tenor}, {swap_tenor})."
            )

        vol_surface.loc[option_tenor, swap_tenor] /= forward_rate

plot_surface(
    vol_surface,
    title="Processed Lognormal-Equivalent Volatility Surface",
)


## 4. Swaption calibration instruments

The calibration grid follows the original project: option tenors from 3Y to 27Y with corresponding swap tenors from 25Y down to 1Y.


In [ ]:
usd_libor_3m = ql.USDLibor(ql.Period(3, ql.Months), yield_curve)

vol_interpolator = make_surface_interpolator(vol_surface)

CalibrationData = namedtuple(
    "CalibrationData",
    ["start", "length", "volatility"],
)

calibration_points = [
    (start, 28 - start)
    for start in range(3, 28)
]

calibration_data = [
    CalibrationData(
        start=start,
        length=length,
        volatility=get_surface_value(
            vol_interpolator,
            option_tenor=start,
            swap_tenor=length,
        ),
    )
    for start, length in calibration_points
]

calibration_df = pd.DataFrame(calibration_data)
display(calibration_df.head())


## 5. Unconstrained Hull–White calibration

Both the mean-reversion parameter and volatility are calibrated using QuantLib's Levenberg–Marquardt optimizer and Jamshidian swaption engine.


In [ ]:
hull_white_model = ql.HullWhite(yield_curve)
jamshidian_engine = ql.JamshidianSwaptionEngine(hull_white_model)

swaptions = create_swaption_helpers(
    calibration_data,
    usd_libor_3m,
    yield_curve,
    jamshidian_engine,
)

optimizer = ql.LevenbergMarquardt(
    1.0e-8,
    1.0e-8,
    1.0e-8,
)

end_criteria = ql.EndCriteria(
    10000,
    100,
    1e-6,
    1e-8,
    1e-8,
)

hull_white_model.calibrate(
    swaptions,
    optimizer,
    end_criteria,
)

mean_reversion, hw_volatility = hull_white_model.params()

print(f"Calibrated mean reversion (a): {mean_reversion:.5f}")
print(f"Calibrated volatility (sigma): {hw_volatility:.5f}")

calibration_report_df = calibration_report(swaptions, calibration_data)
display(calibration_report_df.round(6))


## 6. Fixed mean-reversion calibration diagnostic

A second calibration fixes mean reversion at 0.03 and calibrates only the volatility parameter. This is kept as a diagnostic rather than being used for the final bond valuation.

The original notebook contained a variable-reference bug here: the calibrated parameters were read from the unconstrained model after calibrating a separate model. That bug is corrected below.


In [ ]:
fixed_mean_reversion = 0.03

fixed_reversion_model = ql.HullWhite(
    yield_curve,
    fixed_mean_reversion,
    0.001,
)

fixed_reversion_engine = ql.JamshidianSwaptionEngine(
    fixed_reversion_model
)

fixed_reversion_swaptions = create_swaption_helpers(
    calibration_data,
    usd_libor_3m,
    yield_curve,
    fixed_reversion_engine,
)

fixed_reversion_model.calibrate(
    fixed_reversion_swaptions,
    optimizer,
    end_criteria,
    ql.NoConstraint(),
    [],
    [True, False],
)

fixed_a, fixed_sigma = fixed_reversion_model.params()

print(f"Fixed mean reversion (a): {fixed_a:.5f}")
print(f"Calibrated volatility (sigma): {fixed_sigma:.5f}")

fixed_calibration_report_df = calibration_report(
    fixed_reversion_swaptions,
    calibration_data,
)
display(fixed_calibration_report_df.round(6))


## 7. Callable bond construction

In [ ]:
callability_schedule = ql.CallabilitySchedule()

call_price = 100.0
call_date = first_call_date

number_of_call_dates = int(
    np.ceil((maturity_date - first_call_date) / 365.25)
)

for _ in range(number_of_call_dates):
    callability_price = ql.CallabilityPrice(
        call_price,
        ql.CallabilityPrice.Clean,
    )

    callability_schedule.append(
        ql.Callability(
            callability_price,
            ql.Callability.Call,
            call_date,
        )
    )

    call_date = calendar.advance(
        call_date,
        1,
        ql.Years,
    )

bond_tenor = ql.Period(ql.Annual)

bond_schedule = ql.Schedule(
    issue_date,
    maturity_date,
    bond_tenor,
    calendar,
    business_day_convention,
    business_day_convention,
    ql.DateGeneration.Backward,
    False,
)

settlement_days = 3
face_amount = 100.0
coupon_rate = 0.0425

callable_bond = ql.CallableFixedRateBond(
    settlement_days,
    face_amount,
    bond_schedule,
    [coupon_rate],
    bond_day_counter,
    business_day_convention,
    face_amount,
    issue_date,
    callability_schedule,
)

print(f"Callable bond has {len(callability_schedule)} scheduled call dates.")


## 8. Callable bond valuation

In [ ]:
callable_price = price_callable_bond(
    yield_curve,
    mean_reversion,
    hw_volatility,
    grid_points=40,
    bond=callable_bond,
)

print(f"Callable bond dirty price: {callable_price:.6f}")


## 9. Non-callable benchmark valuation

In [ ]:
non_callable_schedule = ql.Schedule(
    issue_date,
    maturity_date,
    bond_tenor,
    calendar,
    business_day_convention,
    business_day_convention,
    ql.DateGeneration.Backward,
    False,
)

non_callable_bond = ql.FixedRateBond(
    settlement_days,
    face_amount,
    non_callable_schedule,
    [coupon_rate],
    bond_day_counter,
)

non_callable_engine = ql.DiscountingBondEngine(yield_curve)
non_callable_bond.setPricingEngine(non_callable_engine)

non_callable_price = non_callable_bond.dirtyPrice()

embedded_call_value = non_callable_price - callable_price

print(f"Non-callable dirty price: {non_callable_price:.6f}")
print(f"Implied embedded call value: {embedded_call_value:.6f}")


## 10. Interest-rate sensitivity and price compression

The analysis reprices both bonds over a flat-rate range of 2%–7% using 200 rate points. The same calibrated Hull–White parameters are retained while the discount curve is replaced by a flat curve at each scenario rate.


In [ ]:
scenario_rates = np.linspace(0.02, 0.07, 200)

callable_prices = np.empty_like(scenario_rates)
bullet_prices = np.empty_like(scenario_rates)

for i, rate in enumerate(scenario_rates):
    flat_curve = ql.FlatForward(
        valuation_date,
        float(rate),
        curve_day_counter,
        ql.Compounded,
        ql.Annual,
    )
    flat_curve_handle = ql.YieldTermStructureHandle(flat_curve)

    callable_prices[i] = price_callable_bond(
        flat_curve_handle,
        mean_reversion,
        hw_volatility,
        grid_points=200,
        bond=callable_bond,
    )

    flat_discount_engine = ql.DiscountingBondEngine(flat_curve_handle)
    non_callable_bond.setPricingEngine(flat_discount_engine)
    bullet_prices[i] = non_callable_bond.dirtyPrice()

sensitivity_df = pd.DataFrame(
    {
        "Callable": callable_prices,
        "Non-callable": bullet_prices,
        "Embedded Call Value": bullet_prices - callable_prices,
        "Callable Discount (%)": 100.0 * (bullet_prices - callable_prices) / bullet_prices,
    },
    index=scenario_rates,
)

display(sensitivity_df.head())


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(
    scenario_rates * 100,
    callable_prices,
    label="Callable bond",
)
ax.plot(
    scenario_rates * 100,
    bullet_prices,
    label="Non-callable bond",
)

ax.set_xlabel("Flat interest rate (%)")
ax.set_ylabel("Dirty price")
ax.set_title("Callable vs. Non-callable Bond Price Sensitivity")
ax.legend()
ax.grid(alpha=0.25)

plt.show()


## 11. Numerical summary

The following summary makes the main project outputs easy to inspect after execution.


In [ ]:
summary = pd.Series(
    {
        "Hull–White mean reversion": mean_reversion,
        "Hull–White volatility": hw_volatility,
        "Callable bond dirty price": callable_price,
        "Non-callable dirty price": non_callable_price,
        "Embedded call value": embedded_call_value,
        "Maximum callable discount (%)": sensitivity_df["Callable Discount (%)"].max(),
        "Minimum callable discount (%)": sensitivity_df["Callable Discount (%)"].min(),
    }
)

display(summary.to_frame("Value").round(6))
